### Genie Spaces

Creates the Databricks Genie spaces used by the operational dashboard supervisor:

- **Revenue & Orders Intelligence** — real-time revenue, order volume, and location
  performance from Lakeflow gold tables.
- **Operations Intelligence** — kitchen ops, event patterns, location health, and
  food safety signals from Lakeflow + food_safety tables.
- **Menu & Safety Intelligence** — menu items, nutrition, allergens, inspection
  scores, and violations from the menu_documents pipeline.

All spaces share a single SQL warehouse and reuse `create_or_get_genie()` so the
stage is idempotent.

In [ ]:
%pip install --upgrade databricks-sdk

In [ ]:
dbutils.library.restartPython()

In [ ]:
CATALOG = dbutils.widgets.get("CATALOG")

import sys
sys.path.append('../utils')
from ops_profile import (
    parse_csv_keys,
    GENIE_KEYS,
    DEFAULT_GENIE_SPACE_KEYS,
    genie_title,
    GENIE_DOMAIN_TAGS,
)

def _job_param(name: str, default: str = "") -> str:
    try:
        return dbutils.widgets.get(name)
    except Exception:
        return default

GENIE_KEY_LIST = parse_csv_keys(
    _job_param("GENIE_SPACE_KEYS", DEFAULT_GENIE_SPACE_KEYS),
    GENIE_KEYS,
    DEFAULT_GENIE_SPACE_KEYS,
    "GENIE_SPACE_KEYS",
)
print(f"Genie space keys: {GENIE_KEY_LIST}")


##### Create or reuse a SQL warehouse for the Genie spaces

In [ ]:
from databricks.sdk import WorkspaceClient
import json, hashlib, sys

sys.path.append('../utils')
from uc_state import add

w = WorkspaceClient()

_ops_wh = _job_param("OPS_WAREHOUSE_NAME").strip()
WAREHOUSE_NAME = _ops_wh or f"{CATALOG}-genie-warehouse"
if _ops_wh:
    print(f"Using existing SQL warehouse: {WAREHOUSE_NAME!r}")
existing_wh = [wh for wh in w.warehouses.list() if wh.name == WAREHOUSE_NAME]
if existing_wh:
    warehouse = existing_wh[0]
    print(f"\u267b\ufe0f Using existing warehouse: {warehouse.id}")
else:
    warehouse = w.warehouses.create(
        name=WAREHOUSE_NAME,
        cluster_size="Small",
        max_num_clusters=1,
        min_num_clusters=1,
        enable_serverless_compute=True,
    ).result()
    add(CATALOG, "warehouses", warehouse)
    print(f"\u2705 Created warehouse: {warehouse.id}")

##### Helper: create or reuse a Genie space

Looks up existing spaces in `uc_state` by title; only creates a new one when no
matching space is found (or the recorded space has been deleted out-of-band).

In [ ]:
import uuid


def make_questions(questions):
    return [
        {"id": hashlib.md5(q.encode()).hexdigest(), "question": [q]}
        for q in questions
    ]


def _space_warehouse_id(space) -> str | None:
    """Best-effort extraction of the warehouse_id a Genie space is bound to."""
    for attr in ("warehouse_id", "config_warehouse_id"):
        v = getattr(space, attr, None)
        if v:
            return v
    return None


def _delete_genie_space(space_id: str) -> None:
    """Hard-delete a Genie space; tolerate already-gone."""
    try:
        w.api_client.do("DELETE", f"/api/2.0/genie/spaces/{space_id}")
    except Exception as e:
        print(f"⚠️ Failed to delete stale Genie space {space_id}: {e}")


# -----------------------------------------------------------------------------
# serialized_space builders (v1 schema).
#
# Schema sources:
#   - Medium API guide (Dec 2025):  text_instructions + example_question_sqls
#   - Databricks community forum (Jan 6 2026, Databricks Employee response):
#     join_specs and sql_snippets must be nested under `instructions`,
#     NOT under `data_sources`.  This is the format Genie actually persists.
#     https://community.databricks.com/t5/data-engineering/issues-creating-genie-space-via-api-join-specs-are-not-persisted/m-p/143116
#
# Joins use the `--rt=FROM_RELATIONSHIP_TYPE_*--` annotation in the SQL list
# rather than a structured relationship_type field.
# -----------------------------------------------------------------------------


def _build_text_instructions(text: str) -> list[dict]:
    """Genie expects a list of instruction blocks; each block's `content` is a
    list of strings (typically one per line).  Splitting on newlines keeps the
    UI-rendered text readable without forcing the caller to pre-chunk it."""
    lines = text.split("\n")
    return [{
        "id": uuid.uuid4().hex[:32],
        "content": [line + "\n" for line in lines],
    }]


def _build_example_question_sqls(items) -> list[dict]:
    """items: list of dicts with keys 'question' and 'sql'."""
    return [
        {
            "id": uuid.uuid4().hex[:32],
            "question": [item["question"]],
            "sql": [item["sql"]],
        }
        for item in items
    ]


def _build_benchmarks(items) -> list[dict]:
    """Build benchmark question objects for the serialized_space `benchmarks.questions` array.

    items: list of dicts with keys 'question' (str) and 'sql' (str, the ground-truth SQL).

    Schema per the Genie Spaces API reference:
      {
        "id":       <32-char lowercase hex>,
        "question": [<question text>],
        "answer":   [{"format": "SQL", "content": [<sql>]}]
      }

    Notes:
      - `uuid.uuid4().hex` returns a 32-char lowercase hex string (no hyphens),
        which is the format Genie requires.
      - IDs MUST be unique across `config.sample_questions` AND
        `benchmarks.questions` per the API validation rules — we use random
        UUIDs for benchmarks and MD5-of-text for sample questions, so a
        collision would require an MD5 to equal a UUID, which is effectively
        impossible.
      - Each benchmark must have exactly ONE `answer` entry with format=SQL,
        per the API validation rules.
      - Aim for ≥5 benchmarks per space (Databricks best practice).
    """
    return [
        {
            "id": uuid.uuid4().hex,
            "question": [item["question"]],
            "answer": [{"format": "SQL", "content": [item["sql"]]}],
        }
        for item in items
    ]


def _build_join_spec(
    left_identifier: str,
    right_identifier: str,
    on,
    rt: str = "FROM_RELATIONSHIP_TYPE_MANY_TO_ONE",
) -> dict:
    """Build one join_spec object.

    Args:
      left_identifier: fully-qualified left table name.
      right_identifier: fully-qualified right table name.
      on: list of (left_col, right_col) tuples.
      rt: relationship type annotation (MANY_TO_ONE / ONE_TO_MANY / ONE_TO_ONE).
    """
    left_alias = left_identifier.split(".")[-1]
    right_alias = right_identifier.split(".")[-1]
    conds = " AND ".join(
        f"`{left_alias}`.`{lc}` = `{right_alias}`.`{rc}`" for lc, rc in on
    )
    return {
        "id": uuid.uuid4().hex[:32],
        "left": {"identifier": left_identifier, "alias": left_alias},
        "right": {"identifier": right_identifier, "alias": right_alias},
        "sql": [conds, f"--rt={rt}--"],
    }


def _merge_genie_content(
    base: dict,
    tables: list[str],
    instructions_text: str,
    example_sqls: list[dict],
    sample_questions: list[str],
    join_specs: list[dict] | None = None,
    benchmarks: list[dict] | None = None,
) -> dict:
    """Merge our authored content INTO Genie's freshly-exported serialized_space.

    We deliberately mutate the dict Genie just handed us back rather than
    building one from scratch — that's the only way to satisfy Genie's
    "export format" check, which fails with 'Aborted: The export format has
    changed since this export was taken. Re-export the space and merge your
    changes.' if we POST a hand-built body with a stale/missing version field.
    """
    # Genie validates that id-bearing lists are sorted by id; rejects POST
    # with 'InvalidParameterValue: Invalid export proto: <field> must be
    # sorted by id' otherwise.
    def _by_id(items):
        return sorted(items, key=lambda x: x.get("id", ""))

    # REPLACE entire subtrees (don't setdefault/merge): the bootstrap
    # template may carry stale UC function references (function_specs,
    # sql_snippets, certified_answers) under instructions, and stale table
    # refs under data_sources, all pointing at the *previous* catalog.
    # When Genie validates the POST body it'll fail with ROUTINE_NOT_FOUND
    # / TABLE_NOT_FOUND on those.  Wiping the subtrees and writing only what
    # we authored avoids the problem.
    base["data_sources"] = {"tables": [{"identifier": t} for t in sorted(tables)]}
    instructions: dict = {
        "text_instructions": _by_id(_build_text_instructions(instructions_text)),
        "example_question_sqls": _by_id(_build_example_question_sqls(example_sqls)),
    }
    if join_specs:
        instructions["join_specs"] = _by_id(join_specs)
    base["instructions"] = instructions
    base["config"] = {"sample_questions": _by_id(make_questions(sample_questions))}

    # Benchmarks: same wipe-and-replace pattern as the other subtrees.  If the
    # bootstrap template carried stale benchmarks pointing at the previous
    # catalog, leaving them in place would make the POST fail validation.
    # Schema documented in the Genie Spaces API reference under
    # `serialized_space.benchmarks.questions`.
    if benchmarks:
        base["benchmarks"] = {"questions": _by_id(_build_benchmarks(benchmarks))}
    else:
        base.pop("benchmarks", None)

    return base


def _verify_genie_content(space_id: str, sent: dict) -> None:
    """Fetch the space back and log what Genie actually persisted vs sent.

    Genie's serialized_space schema is sparsely documented and the API
    silently drops fields it doesn't recognise — so a "successful" create
    can still produce an empty-looking space in the UI.  This diagnostic
    prints counts on every run so the task log is the source of truth.

    Note: get_space() does NOT include serialized_space by default; you must
    pass include_serialized_space=True (requires CAN EDIT on the space).
    """
    try:
        fresh = w.genie.get_space(space_id, include_serialized_space=True)
    except Exception as e:
        print(f"  \u26a0\ufe0f verify: could not get_space {space_id}: {e}")
        return
    raw = getattr(fresh, "serialized_space", None)
    if not raw:
        print(f"  \u26a0\ufe0f verify: get_space returned no serialized_space for {space_id}")
        return
    try:
        got = json.loads(raw) if isinstance(raw, str) else raw
    except Exception as e:
        print(f"  \u26a0\ufe0f verify: serialized_space not JSON: {e}")
        return

    got_ins = got.get("instructions") or {}
    got_text = got_ins.get("text_instructions") or []
    got_examples = got_ins.get("example_question_sqls") or []
    got_joins = got_ins.get("join_specs") or []
    got_tables = (got.get("data_sources") or {}).get("tables") or []
    got_questions = (got.get("config") or {}).get("sample_questions") or []
    got_benchmarks = (got.get("benchmarks") or {}).get("questions") or []

    print(
        f"  \U0001F50E verify {space_id}: "
        f"tables sent={sent['tables']} got={len(got_tables)} | "
        f"text_instructions sent={sent['text']} got={len(got_text)} | "
        f"example_sqls sent={sent['examples']} got={len(got_examples)} | "
        f"join_specs sent={sent['joins']} got={len(got_joins)} | "
        f"sample_questions sent={sent['questions']} got={len(got_questions)} | "
        f"benchmarks sent={sent['benchmarks']} got={len(got_benchmarks)}"
    )

    missing = []
    if sent["text"] and not got_text:
        missing.append("text_instructions")
    if sent["examples"] and not got_examples:
        missing.append("example_question_sqls")
    if sent["joins"] and not got_joins:
        missing.append("join_specs")
    if sent["benchmarks"] and not got_benchmarks:
        missing.append("benchmarks")
    if missing:
        print(
            f"  \u26a0\ufe0f Genie did NOT persist {missing} for {space_id} — "
            f"schema mismatch.  Dumping what Genie returned for debugging:"
        )
        print("  " + json.dumps(got, indent=2)[:4000])


# Identity-bearing fields that vary per-space; we must strip these before
# reusing a template's serialized_space envelope to create a new space.
_TEMPLATE_IDENTITY_FIELDS = (
    "id", "space_id", "title", "description",
    "warehouse_id", "created_at", "updated_at",
    "creator_id", "creator", "owner", "owner_id",
    "etag", "_etag",
)


def _bootstrap_template_serialized_space(skip_space_ids: set | None = None) -> dict | None:
    """Find any existing Genie space and pull its serialized_space as a template.

    Why this is necessary:
      The SDK docs (databricks/sdk/service/dashboards.py, create_space)
      explicitly say to "Use the Get Genie Space API to retrieve an example
      response, which includes the serialized_space field" — i.e. the
      serialized_space format is opaque (carries server-managed version
      metadata) and hand-built bodies are rejected with "Aborted: The export
      format has changed since this export was taken."

      So we bootstrap by listing existing spaces in the workspace and
      copying one Genie produced, then mutate it before reusing.

    Returns the parsed dict (envelope to mutate) or None if no usable
    template was found (caller should fall back to serialized_space=None).
    """
    skip = skip_space_ids or set()
    try:
        resp = w.genie.list_spaces()
    except Exception as e:
        print(f"  \u26a0\ufe0f list_spaces failed: {e}")
        return None
    spaces = getattr(resp, "spaces", None) or []
    for entry in spaces:
        sid = getattr(entry, "space_id", None)
        if not sid or sid in skip:
            continue
        try:
            fresh = w.genie.get_space(sid, include_serialized_space=True)
        except Exception:
            # No CAN EDIT permission, or transient — skip this candidate.
            continue
        raw = getattr(fresh, "serialized_space", None)
        if not raw:
            continue
        try:
            parsed = json.loads(raw)
        except Exception:
            continue
        if isinstance(parsed, dict) and parsed:
            print(f"  \U0001F4CB Using existing space {sid} as serialized_space template")
            return parsed
    return None


def _scrub_template(envelope: dict) -> dict:
    """Remove identity-bearing fields from a template envelope before reusing
    it to create a new space.  Returns the same dict (mutated in place)."""
    for key in _TEMPLATE_IDENTITY_FIELDS:
        envelope.pop(key, None)
    return envelope


def create_or_get_genie(
    title,
    description,
    tables,
    instructions_text,
    example_sqls,
    sample_questions,
    join_specs=None,
    benchmarks=None,
):
    """Recreate a Genie space, bootstrapping serialized_space from an existing one.

    Why this pattern:
      Genie validates serialized_space against an opaque server-managed
      format version.  Hand-built bodies (including "{}") are rejected with
      'Aborted: The export format has changed since this export was taken.
      Re-export the space and merge your changes.'  The SDK docs say to
      bootstrap from a real GET response.

    Flow:
      1. Delete any pre-existing space(s) with this title (uc_state lookup).
      2. Bootstrap a serialized_space template by listing existing workspace
         spaces and pulling one with include_serialized_space=True.
      3. Inject our content (tables, instructions, joins, sample questions)
         into that template, strip identity fields.
      4. create_space with the modified template.
      5. GET the new space (include_serialized_space=True) and re-merge our
         content into the *post-create* envelope (server may have normalized
         our submission), then update_space.
      6. Verify what actually persisted.

      Fallback: if no template space exists in the workspace, attempt
      create_space(serialized_space=None) — the SDK omits the field, and
      the API may create with defaults; in that case we still GET+update.

    space_id changes on every run; downstream resolves via uc_state
    (ORDER BY created_at DESC).
    """
    # Step 1: Collect old space_ids for this title so we can both delete them
    # AND skip them when looking for a template (don't bootstrap from the
    # exact space we're about to delete).
    #
    # Two passes:
    #   (a) uc_state lookup — fast, authoritative for spaces this stage owns.
    #   (b) Workspace-wide title sweep — Genie spaces are *workspace-scoped*,
    #       not catalog-scoped, so a previous `bundle destroy` (which wipes
    #       uc_state) leaves orphan spaces alive in the workspace.  Without
    #       this sweep, the next create_space + update_space round-trip dies
    #       with `AlreadyExists: Node named '<title>' already exists` because
    #       Genie sees the title collision against the orphan.
    old_space_ids: set[str] = set()

    # (a) uc_state-tracked spaces
    try:
        df = spark.sql(f"""
            SELECT resource_data FROM {CATALOG}._internal_state.resources
            WHERE resource_type = 'genie_spaces'
            ORDER BY created_at DESC
        """)
        for row in df.collect():
            info = json.loads(row.resource_data)
            if info.get("title") != title:
                continue
            sid = info.get("space_id")
            if not sid:
                continue
            try:
                w.genie.get_space(sid)
            except Exception:
                continue
            old_space_ids.add(sid)
            print(f"\U0001F5D1\uFE0F  Deleting pre-existing Genie space {sid} from uc_state (title={title!r})")
            _delete_genie_space(sid)
    except Exception as e:
        print(f"\u26a0\ufe0f Could not enumerate existing Genie spaces from uc_state: {e}")

    # (b) Workspace-wide orphan sweep — catches spaces left behind by previous
    # `bundle destroy` runs.  We must drain the paginated list_spaces output
    # because a workspace can easily accumulate dozens of stale demo spaces.
    try:
        page_token = None
        scanned = 0
        while True:
            resp = w.genie.list_spaces(page_token=page_token) if page_token else w.genie.list_spaces()
            for entry in (getattr(resp, "spaces", None) or []):
                scanned += 1
                if getattr(entry, "title", None) != title:
                    continue
                sid = getattr(entry, "space_id", None)
                if not sid or sid in old_space_ids:
                    continue
                print(f"\U0001F5D1\uFE0F  Deleting orphan Genie space {sid} from workspace (title={title!r}, not in uc_state)")
                _delete_genie_space(sid)
                old_space_ids.add(sid)
            page_token = getattr(resp, "next_page_token", None)
            if not page_token:
                break
        if scanned:
            print(f"  \U0001F50E workspace sweep scanned {scanned} Genie spaces")
    except Exception as e:
        print(f"\u26a0\ufe0f Workspace-wide Genie space sweep failed: {e}")

    # Step 2: Bootstrap a template from any other existing space.
    template = _bootstrap_template_serialized_space(skip_space_ids=old_space_ids)

    # Step 3: Build initial serialized_space.
    if template is not None:
        _scrub_template(template)
        merged = _merge_genie_content(
            template,
            tables=tables,
            instructions_text=instructions_text,
            example_sqls=example_sqls,
            sample_questions=sample_questions,
            join_specs=join_specs,
            benchmarks=benchmarks,
        )
        initial_serialized = json.dumps(merged)
    else:
        print("  \u26a0\ufe0f No usable Genie space template found; trying create_space with serialized_space=None")
        initial_serialized = None

    # Step 4: Create.
    try:
        space = w.genie.create_space(
            warehouse_id=warehouse.id,
            serialized_space=initial_serialized,
            title=title,
            description=description,
        )
    except Exception as e:
        print(f"\u274c create_space failed: {e}")
        if initial_serialized is None:
            print(
                "   Hint: no existing Genie space was found to bootstrap from. "
                "Create ONE space manually in the workspace UI (any catalog) and rerun this task — "
                "the API requires a real serialized_space template to copy from."
            )
        else:
            print(f"   Template (first 2000 chars):\n   {initial_serialized[:2000]}")
        raise
    genie_space_id = space.space_id
    print(f"\u2705 Created Genie space: {genie_space_id}")
    add(CATALOG, "genie_spaces", {"space_id": genie_space_id, "title": title})

    # Step 5: GET the new space, re-merge content, push back via update_space.
    # We do this even on the template path because (a) the server may have
    # normalized our submission, and (b) if we created with None, this is
    # the first time we get to inject content at all.
    try:
        fresh = w.genie.get_space(genie_space_id, include_serialized_space=True)
        raw = getattr(fresh, "serialized_space", None)
    except Exception as e:
        print(f"\u26a0\ufe0f get_space after create failed: {e}; skipping content update")
        raw = None

    if raw:
        try:
            envelope = json.loads(raw)
        except Exception as e:
            print(f"\u26a0\ufe0f Could not parse post-create serialized_space ({e})")
            envelope = {}
        merged = _merge_genie_content(
            envelope,
            tables=tables,
            instructions_text=instructions_text,
            example_sqls=example_sqls,
            sample_questions=sample_questions,
            join_specs=join_specs,
            benchmarks=benchmarks,
        )
        try:
            w.genie.update_space(
                space_id=genie_space_id,
                serialized_space=json.dumps(merged),
                title=title,
                description=description,
                warehouse_id=warehouse.id,
            )
            print(f"\u2705 Pushed content to Genie space {genie_space_id}")
        except Exception as e:
            print(f"\u274c update_space failed: {e}")
            print(f"   Genie's fresh export (first 2000 chars):\n   {raw[:2000]}")
            print(f"   Our merged version (first 2000 chars):\n   {json.dumps(merged, indent=2)[:2000]}")
            raise

    _verify_genie_content(
        genie_space_id,
        sent={
            "tables": len(tables),
            "text": 1 if instructions_text else 0,
            "examples": len(example_sqls or []),
            "joins": len(join_specs or []),
            "questions": len(sample_questions or []),
            "benchmarks": len(benchmarks or []),
        },
    )

    return genie_space_id

##### Revenue & Orders Intelligence Genie

In [ ]:
revenue_genie_id = None
if "revenue" in GENIE_KEY_LIST:
    REVENUE_TITLE = f"Revenue & Orders Intelligence ({CATALOG})"

    REVENUE_DESCRIPTION = (
        "Revenue and order analytics for Casper's Kitchens. "
        "Uses pre-aggregated Lakeflow gold tables for fast queries on per-location, "
        "per-brand, and per-order revenue, joined to simulator dimensions for friendly names."
    )

    # General instructions — kept under ~20 lines per Genie best practice.
    REVENUE_INSTRUCTIONS = """You are a revenue analytics assistant for Casper's Kitchens, a ghost-kitchen network.
    Always prefer pre-aggregated Gold tables — they are 10-100x faster than scanning all_events.

    Table routing (smallest table that can answer wins):
    - Revenue / orders per location, time-series  -> gold_location_sales_hourly JOIN simulator.locations
    - Revenue / orders per brand, daily           -> gold_brand_sales_day JOIN simulator.brands
    - Average order value, per-order metrics      -> gold_order_header JOIN simulator.locations
    - Item-level detail                            -> silver_order_items
    - Raw event detail (LAST RESORT)               -> all_events (1M+ rows, JSON body)

    Conventions:
    - Revenue columns are already SUM-able: revenue (gold_location_sales_hourly), brand_revenue (gold_brand_sales_day), order_revenue (gold_order_header).
    - "today" = hour_ts >= current_date(); "this week" = hour_ts >= current_timestamp() - INTERVAL 7 DAYS.
    - For top-N queries default to LIMIT 10 unless the user specifies otherwise.
    - Always JOIN simulator.locations on location_id to return readable location names.
    - Always JOIN simulator.brands on brand_id to return readable brand names.
    - Always filter by a date range to reduce scan cost."""

    REVENUE_TABLES = [
        f"{CATALOG}.lakeflow.gold_location_sales_hourly",
        f"{CATALOG}.lakeflow.gold_brand_sales_day",
        f"{CATALOG}.lakeflow.gold_order_header",
        f"{CATALOG}.lakeflow.silver_order_items",
        f"{CATALOG}.lakeflow.all_events",
        f"{CATALOG}.simulator.locations",
        f"{CATALOG}.simulator.brands",
        f"{CATALOG}.simulator.items",
        f"{CATALOG}.simulator.brand_locations",
    ]

    REVENUE_QUESTIONS = [
        "What is total revenue by location for the last 30 days?",
        "Which brand generates the most orders across all locations?",
        "What is the order cancellation rate by location?",
        "Show me hourly order volume trends for today",
        "Which location has the highest average order value?",
        "What is the revenue split between locations this week vs last week?",
        "How many total orders were placed across all locations?",
        "Which brands are underperforming relative to the network average?",
    ]

    REVENUE_EXAMPLE_SQLS = [
        {
            "question": "What is total revenue by location for the last 30 days?",
            "sql": f"""SELECT l.name AS location, SUM(g.revenue) AS revenue_30d, SUM(g.orders) AS orders_30d
    FROM {CATALOG}.lakeflow.gold_location_sales_hourly g
    JOIN {CATALOG}.simulator.locations l ON g.location_id = l.location_id
    WHERE g.hour_ts >= current_timestamp() - INTERVAL 30 DAYS
    GROUP BY l.name
    ORDER BY revenue_30d DESC""",
        },
        {
            "question": "Which brand generated the most orders in the last 7 days?",
            "sql": f"""SELECT b.name AS brand, SUM(g.orders) AS orders_7d, SUM(g.brand_revenue) AS revenue_7d
    FROM {CATALOG}.lakeflow.gold_brand_sales_day g
    JOIN {CATALOG}.simulator.brands b ON g.brand_id = b.brand_id
    WHERE g.day >= current_date() - 7
    GROUP BY b.name
    ORDER BY orders_7d DESC
    LIMIT 10""",
        },
        {
            "question": "Average order value by location, last 30 days",
            "sql": f"""SELECT l.name AS location, ROUND(AVG(h.order_revenue), 2) AS avg_order_value, COUNT(*) AS orders
    FROM {CATALOG}.lakeflow.gold_order_header h
    JOIN {CATALOG}.simulator.locations l ON h.location_id = l.location_id
    WHERE h.order_day >= current_date() - 30
    GROUP BY l.name
    ORDER BY avg_order_value DESC""",
        },
        {
            "question": "Show hourly order volume trends for today",
            "sql": f"""SELECT g.hour_ts, l.name AS location, g.orders, g.revenue
    FROM {CATALOG}.lakeflow.gold_location_sales_hourly g
    JOIN {CATALOG}.simulator.locations l ON g.location_id = l.location_id
    WHERE g.hour_ts >= current_date()
    ORDER BY g.hour_ts, l.name""",
        },
        {
            "question": "Revenue this week vs last week, by location",
            "sql": f"""SELECT l.name AS location,
           SUM(CASE WHEN g.hour_ts >= current_timestamp() - INTERVAL 7 DAYS THEN g.revenue ELSE 0 END) AS revenue_this_week,
           SUM(CASE WHEN g.hour_ts <  current_timestamp() - INTERVAL 7 DAYS
                    AND  g.hour_ts >= current_timestamp() - INTERVAL 14 DAYS THEN g.revenue ELSE 0 END) AS revenue_prev_week
    FROM {CATALOG}.lakeflow.gold_location_sales_hourly g
    JOIN {CATALOG}.simulator.locations l ON g.location_id = l.location_id
    WHERE g.hour_ts >= current_timestamp() - INTERVAL 14 DAYS
    GROUP BY l.name
    ORDER BY revenue_this_week DESC""",
        },
        {
            "question": "Top 10 brands by revenue, last 30 days",
            "sql": f"""SELECT b.name AS brand, b.cuisine, SUM(g.brand_revenue) AS revenue_30d, SUM(g.orders) AS orders_30d
    FROM {CATALOG}.lakeflow.gold_brand_sales_day g
    JOIN {CATALOG}.simulator.brands b ON g.brand_id = b.brand_id
    WHERE g.day >= current_date() - 30
    GROUP BY b.name, b.cuisine
    ORDER BY revenue_30d DESC
    LIMIT 10""",
        },
    ]

    REVENUE_JOIN_SPECS = [
        _build_join_spec(
            f"{CATALOG}.lakeflow.gold_location_sales_hourly",
            f"{CATALOG}.simulator.locations",
            on=[("location_id", "location_id")],
        ),
        _build_join_spec(
            f"{CATALOG}.lakeflow.gold_brand_sales_day",
            f"{CATALOG}.simulator.brands",
            on=[("brand_id", "brand_id")],
        ),
        _build_join_spec(
            f"{CATALOG}.lakeflow.gold_order_header",
            f"{CATALOG}.simulator.locations",
            on=[("location_id", "location_id")],
        ),
        _build_join_spec(
            f"{CATALOG}.lakeflow.silver_order_items",
            f"{CATALOG}.simulator.locations",
            on=[("location_id", "location_id")],
        ),
        _build_join_spec(
            f"{CATALOG}.lakeflow.silver_order_items",
            f"{CATALOG}.simulator.brands",
            on=[("brand_id", "brand_id")],
        ),
    ]

    # Benchmark questions — ground-truth SQL pairs the presenter (or the
    # `Genie_Spaces` task) can rerun via the Genie UI's "Run Benchmarks" button
    # (or programmatically via `w.genie.genie_create_eval_run(space_id)`) to
    # regression-test the space whenever instructions / example SQLs change.
    #
    # Authoring guidance (per Databricks "Test and monitor a Genie Space" docs):
    #   - aim for ≥5 questions per space (max 500),
    #   - include 2-3 phrasings of the same question to test paraphrase robustness,
    #   - each answer must be a single `format: SQL` entry — Genie compares result
    #     sets cell-by-cell (4-sig-fig numeric tolerance, sort-insensitive).
    #
    # Mix here: 3 questions identical to the example SQLs above (tests that Genie
    # reuses the worked example) + 3 alternate phrasings / simple lookups.
    REVENUE_BENCHMARKS = [
        {
            "question": "What is total revenue by location for the last 30 days?",
            "sql": f"""SELECT l.name AS location, SUM(g.revenue) AS revenue_30d, SUM(g.orders) AS orders_30d
    FROM {CATALOG}.lakeflow.gold_location_sales_hourly g
    JOIN {CATALOG}.simulator.locations l ON g.location_id = l.location_id
    WHERE g.hour_ts >= current_timestamp() - INTERVAL 30 DAYS
    GROUP BY l.name
    ORDER BY revenue_30d DESC""",
        },
        {
            "question": "Show me revenue per location for the past month",
            "sql": f"""SELECT l.name AS location, SUM(g.revenue) AS revenue_30d
    FROM {CATALOG}.lakeflow.gold_location_sales_hourly g
    JOIN {CATALOG}.simulator.locations l ON g.location_id = l.location_id
    WHERE g.hour_ts >= current_timestamp() - INTERVAL 30 DAYS
    GROUP BY l.name
    ORDER BY revenue_30d DESC""",
        },
        {
            "question": "Which brand had the most orders in the past week?",
            "sql": f"""SELECT b.name AS brand, SUM(g.orders) AS orders_7d
    FROM {CATALOG}.lakeflow.gold_brand_sales_day g
    JOIN {CATALOG}.simulator.brands b ON g.brand_id = b.brand_id
    WHERE g.day >= current_date() - 7
    GROUP BY b.name
    ORDER BY orders_7d DESC
    LIMIT 1""",
        },
        {
            "question": "Average order value per location over the last 30 days",
            "sql": f"""SELECT l.name AS location, ROUND(AVG(h.order_revenue), 2) AS avg_order_value
    FROM {CATALOG}.lakeflow.gold_order_header h
    JOIN {CATALOG}.simulator.locations l ON h.location_id = l.location_id
    WHERE h.order_day >= current_date() - 30
    GROUP BY l.name
    ORDER BY avg_order_value DESC""",
        },
        {
            "question": "Top 10 brands by revenue in the last 30 days",
            "sql": f"""SELECT b.name AS brand, SUM(g.brand_revenue) AS revenue_30d
    FROM {CATALOG}.lakeflow.gold_brand_sales_day g
    JOIN {CATALOG}.simulator.brands b ON g.brand_id = b.brand_id
    WHERE g.day >= current_date() - 30
    GROUP BY b.name
    ORDER BY revenue_30d DESC
    LIMIT 10""",
        },
        {
            "question": "How many orders today, per location?",
            "sql": f"""SELECT l.name AS location, SUM(g.orders) AS orders_today
    FROM {CATALOG}.lakeflow.gold_location_sales_hourly g
    JOIN {CATALOG}.simulator.locations l ON g.location_id = l.location_id
    WHERE g.hour_ts >= current_date()
    GROUP BY l.name
    ORDER BY orders_today DESC""",
        },
    ]

    revenue_genie_id = create_or_get_genie(
        title=REVENUE_TITLE,
        description=REVENUE_DESCRIPTION,
        tables=REVENUE_TABLES,
        instructions_text=REVENUE_INSTRUCTIONS,
        example_sqls=REVENUE_EXAMPLE_SQLS,
        sample_questions=REVENUE_QUESTIONS,
        join_specs=REVENUE_JOIN_SPECS,
        benchmarks=REVENUE_BENCHMARKS,
    )
    print(f"Revenue Genie ID: {revenue_genie_id} ({len(REVENUE_BENCHMARKS)} benchmarks)")


##### Operations Intelligence Genie

In [ ]:
ops_genie_id = None
if "ops" in GENIE_KEY_LIST:
    OPS_TITLE = f"Operations Intelligence ({CATALOG})"

    OPS_DESCRIPTION = (
        "Operations analytics for Casper's Kitchens. Covers order throughput, "
        "cancellation rates, food safety inspections, and violations. Combines "
        "Lakeflow gold/silver tables with food_safety inspection data."
    )

    # General instructions — kept under ~20 lines per Genie best practice.
    OPS_INSTRUCTIONS = """You are an operations analytics assistant for Casper's Kitchens (a ghost-kitchen network).
    Prefer Gold/Silver tables over all_events (10-100x faster). all_events is ONLY for derived metrics like cancel rate.

    Table routing:
    - Order volume, throughput, peak hours    -> gold_location_sales_hourly JOIN simulator.locations
    - Per-order ops metrics                    -> gold_order_header
    - Item-level ops                            -> silver_order_items
    - Food safety score / grade by location   -> food_safety.inspections JOIN simulator.locations
    - Specific violations (severity, code)     -> food_safety.violations JOIN food_safety.inspections
    - Cancellation rate (derived from events)  -> all_events with the cancel-rate pattern below

    Cancellation rate computation (orders with order_created but no delivered):
      - total = COUNT(DISTINCT order_id WHERE event_type='order_created')
      - cancelled = total - COUNT(DISTINCT order_id WHERE event_type='delivered')
      - cancel_rate_pct = ROUND(100.0 * cancelled / NULLIF(total, 0), 1)
      - Always filter by event_ts >= current_timestamp() - INTERVAL 30 DAYS and JOIN simulator.locations.

    Conventions:
    - Always JOIN simulator.locations on location_id for readable names.
    - "this week" = ts >= current_timestamp() - INTERVAL 7 DAYS.
    - For cancel rate, ALWAYS run the pattern above — do NOT return 0 without computing it."""

    OPS_TABLES = [
        f"{CATALOG}.lakeflow.gold_location_sales_hourly",
        f"{CATALOG}.lakeflow.gold_order_header",
        f"{CATALOG}.lakeflow.silver_order_items",
        f"{CATALOG}.lakeflow.all_events",
        f"{CATALOG}.simulator.locations",
        f"{CATALOG}.food_safety.inspections",
        f"{CATALOG}.food_safety.violations",
    ]

    OPS_QUESTIONS = [
        "Which locations have the highest complaint rate?",
        "What is the refund approval rate by location?",
        "Show me locations with critical food safety violations",
        "What are the top 5 complaint categories across all locations?",
        "Which location needs the most operational attention right now?",
        "Show order cancellation trends over the last 7 days by location",
        "What is the average food safety inspection score by location?",
        "How many complaints were submitted in the last 24 hours?",
    ]

    OPS_EXAMPLE_SQLS = [
        {
            "question": "Order cancellation rate by location, last 30 days",
            "sql": f"""SELECT l.name AS location,
           COUNT(DISTINCT CASE WHEN ae.event_type = 'order_created' THEN ae.order_id END) AS total_orders,
           COUNT(DISTINCT CASE WHEN ae.event_type = 'order_created' THEN ae.order_id END)
             - COUNT(DISTINCT CASE WHEN ae.event_type = 'delivered' THEN ae.order_id END) AS cancelled_orders,
           ROUND(100.0 * (
             COUNT(DISTINCT CASE WHEN ae.event_type = 'order_created' THEN ae.order_id END)
             - COUNT(DISTINCT CASE WHEN ae.event_type = 'delivered' THEN ae.order_id END)
           ) / NULLIF(COUNT(DISTINCT CASE WHEN ae.event_type = 'order_created' THEN ae.order_id END), 0), 1) AS cancel_rate_pct
    FROM {CATALOG}.lakeflow.all_events ae
    JOIN {CATALOG}.simulator.locations l ON ae.location_id = l.location_id
    WHERE ae.event_ts >= current_timestamp() - INTERVAL 30 DAYS
    GROUP BY l.name
    ORDER BY cancel_rate_pct DESC""",
        },
        {
            "question": "Average food safety score by location",
            "sql": f"""SELECT l.name AS location,
           ROUND(AVG(i.score), 1) AS avg_score,
           MIN(i.score) AS worst_score,
           MAX(i.inspection_date) AS latest_inspection
    FROM {CATALOG}.food_safety.inspections i
    JOIN {CATALOG}.simulator.locations l ON i.location_id = l.location_id
    GROUP BY l.name
    ORDER BY avg_score""",
        },
        {
            "question": "Critical violations needing immediate action in the last 60 days",
            "sql": f"""SELECT l.name AS location, v.inspection_date, v.code, v.category, v.severity, v.description
    FROM {CATALOG}.food_safety.violations v
    JOIN {CATALOG}.simulator.locations l ON v.location_id = l.location_id
    WHERE v.severity = 'critical'
      AND v.inspection_date >= current_date() - 60
    ORDER BY v.inspection_date DESC, l.name""",
        },
        {
            "question": "Hourly order volume per location for today",
            "sql": f"""SELECT g.hour_ts, l.name AS location, g.orders, g.revenue
    FROM {CATALOG}.lakeflow.gold_location_sales_hourly g
    JOIN {CATALOG}.simulator.locations l ON g.location_id = l.location_id
    WHERE g.hour_ts >= current_date()
    ORDER BY g.hour_ts, l.name""",
        },
        {
            "question": "Locations with the most recent failing inspection (score < 70)",
            "sql": f"""SELECT l.name AS location, i.inspection_date, i.score, i.grade, i.violation_count, i.critical_count
    FROM {CATALOG}.food_safety.inspections i
    JOIN {CATALOG}.simulator.locations l ON i.location_id = l.location_id
    WHERE i.score < 70
    ORDER BY i.inspection_date DESC
    LIMIT 20""",
        },
        {
            "question": "Daily order trend by location for the last 7 days",
            "sql": f"""SELECT DATE(g.hour_ts) AS day, l.name AS location,
           SUM(g.orders) AS orders,
           ROUND(SUM(g.revenue), 2) AS revenue
    FROM {CATALOG}.lakeflow.gold_location_sales_hourly g
    JOIN {CATALOG}.simulator.locations l ON g.location_id = l.location_id
    WHERE g.hour_ts >= current_timestamp() - INTERVAL 7 DAYS
    GROUP BY DATE(g.hour_ts), l.name
    ORDER BY day, l.name""",
        },
    ]

    OPS_JOIN_SPECS = [
        _build_join_spec(
            f"{CATALOG}.lakeflow.gold_location_sales_hourly",
            f"{CATALOG}.simulator.locations",
            on=[("location_id", "location_id")],
        ),
        _build_join_spec(
            f"{CATALOG}.lakeflow.gold_order_header",
            f"{CATALOG}.simulator.locations",
            on=[("location_id", "location_id")],
        ),
        _build_join_spec(
            f"{CATALOG}.lakeflow.silver_order_items",
            f"{CATALOG}.simulator.locations",
            on=[("location_id", "location_id")],
        ),
        _build_join_spec(
            f"{CATALOG}.lakeflow.all_events",
            f"{CATALOG}.simulator.locations",
            on=[("location_id", "location_id")],
        ),
        _build_join_spec(
            f"{CATALOG}.food_safety.inspections",
            f"{CATALOG}.simulator.locations",
            on=[("location_id", "location_id")],
        ),
        _build_join_spec(
            f"{CATALOG}.food_safety.violations",
            f"{CATALOG}.food_safety.inspections",
            on=[("inspection_id", "inspection_id")],
        ),
        _build_join_spec(
            f"{CATALOG}.food_safety.violations",
            f"{CATALOG}.simulator.locations",
            on=[("location_id", "location_id")],
        ),
    ]

    # Benchmark questions — see REVENUE_BENCHMARKS for authoring notes.
    # Mix: 3 exact matches with example SQLs (paraphrase robustness sanity) +
    # 3 alternate phrasings.  Cancel-rate is the highest-value benchmark because
    # the SQL pattern is non-trivial (nested COUNT DISTINCT + NULLIF) and the
    # example SQL is what teaches it — if the benchmark regresses, the cancel-rate
    # example was likely dropped or paraphrased badly.
    OPS_BENCHMARKS = [
        {
            "question": "What is the order cancellation rate by location over the last 30 days?",
            "sql": f"""SELECT l.name AS location,
           COUNT(DISTINCT CASE WHEN ae.event_type = 'order_created' THEN ae.order_id END) AS total_orders,
           COUNT(DISTINCT CASE WHEN ae.event_type = 'order_created' THEN ae.order_id END)
             - COUNT(DISTINCT CASE WHEN ae.event_type = 'delivered' THEN ae.order_id END) AS cancelled_orders,
           ROUND(100.0 * (
             COUNT(DISTINCT CASE WHEN ae.event_type = 'order_created' THEN ae.order_id END)
             - COUNT(DISTINCT CASE WHEN ae.event_type = 'delivered' THEN ae.order_id END)
           ) / NULLIF(COUNT(DISTINCT CASE WHEN ae.event_type = 'order_created' THEN ae.order_id END), 0), 1) AS cancel_rate_pct
    FROM {CATALOG}.lakeflow.all_events ae
    JOIN {CATALOG}.simulator.locations l ON ae.location_id = l.location_id
    WHERE ae.event_ts >= current_timestamp() - INTERVAL 30 DAYS
    GROUP BY l.name
    ORDER BY cancel_rate_pct DESC""",
        },
        {
            "question": "Which location is cancelling the most orders?",
            "sql": f"""SELECT l.name AS location,
           ROUND(100.0 * (
             COUNT(DISTINCT CASE WHEN ae.event_type = 'order_created' THEN ae.order_id END)
             - COUNT(DISTINCT CASE WHEN ae.event_type = 'delivered' THEN ae.order_id END)
           ) / NULLIF(COUNT(DISTINCT CASE WHEN ae.event_type = 'order_created' THEN ae.order_id END), 0), 1) AS cancel_rate_pct
    FROM {CATALOG}.lakeflow.all_events ae
    JOIN {CATALOG}.simulator.locations l ON ae.location_id = l.location_id
    WHERE ae.event_ts >= current_timestamp() - INTERVAL 30 DAYS
    GROUP BY l.name
    ORDER BY cancel_rate_pct DESC
    LIMIT 1""",
        },
        {
            "question": "Average food safety inspection score per location",
            "sql": f"""SELECT l.name AS location,
           ROUND(AVG(i.score), 1) AS avg_score
    FROM {CATALOG}.food_safety.inspections i
    JOIN {CATALOG}.simulator.locations l ON i.location_id = l.location_id
    GROUP BY l.name
    ORDER BY avg_score""",
        },
        {
            "question": "Show me critical violations from the last 60 days that need immediate action",
            "sql": f"""SELECT l.name AS location, v.inspection_date, v.code, v.category, v.severity, v.description
    FROM {CATALOG}.food_safety.violations v
    JOIN {CATALOG}.simulator.locations l ON v.location_id = l.location_id
    WHERE v.severity = 'critical'
      AND v.inspection_date >= current_date() - 60
    ORDER BY v.inspection_date DESC, l.name""",
        },
        {
            "question": "Which locations have a failing inspection score (below 70)?",
            "sql": f"""SELECT l.name AS location, i.inspection_date, i.score, i.grade, i.violation_count, i.critical_count
    FROM {CATALOG}.food_safety.inspections i
    JOIN {CATALOG}.simulator.locations l ON i.location_id = l.location_id
    WHERE i.score < 70
    ORDER BY i.inspection_date DESC
    LIMIT 20""",
        },
        {
            "question": "Daily order count per location for the last 7 days",
            "sql": f"""SELECT DATE(g.hour_ts) AS day, l.name AS location,
           SUM(g.orders) AS orders
    FROM {CATALOG}.lakeflow.gold_location_sales_hourly g
    JOIN {CATALOG}.simulator.locations l ON g.location_id = l.location_id
    WHERE g.hour_ts >= current_timestamp() - INTERVAL 7 DAYS
    GROUP BY DATE(g.hour_ts), l.name
    ORDER BY day, l.name""",
        },
    ]

    ops_genie_id = create_or_get_genie(
        title=OPS_TITLE,
        description=OPS_DESCRIPTION,
        tables=OPS_TABLES,
        instructions_text=OPS_INSTRUCTIONS,
        example_sqls=OPS_EXAMPLE_SQLS,
        sample_questions=OPS_QUESTIONS,
        join_specs=OPS_JOIN_SPECS,
        benchmarks=OPS_BENCHMARKS,
    )
    print(f"Operations Genie ID: {ops_genie_id} ({len(OPS_BENCHMARKS)} benchmarks)")


##### Menu & Safety Intelligence Genie

In [ ]:
menu_genie_id = None
if "menu" in GENIE_KEY_LIST:
    MENU_TITLE = f"Menu & Safety Intelligence ({CATALOG})"

    MENU_DESCRIPTION = (
        "Menu and food safety analytics for Casper's Kitchens (16 brands across 4 locations). "
        "Provides menu items, nutrition, allergens, inspection scores, and violation details "
        "from the menu_documents medallion pipeline."
    )

    # General instructions — kept under ~20 lines per Genie best practice.
    MENU_INSTRUCTIONS = f"""You are a menu & food safety analyst for Casper's Kitchens (16 brands, 4 locations).
    All tables live in {CATALOG}.menu_documents. Use gold tables for aggregates, silver for deep dives.

    Table routing:
    - Menu catalog, price tier, allergen flags  -> menu_items + allergens (JOIN on item_name + brand_name)
    - Per-item nutrition deep dive               -> nutritional_info
    - Per-brand nutrition summary                -> brand_nutrition_summary
    - Inspection scores, grades, severity        -> inspection_details OR silver_inspections
    - Individual violations + urgency            -> violation_analysis OR silver_violations
    - Per-location pass rate, severity index     -> location_compliance_summary

    Domain conventions:
    - price_tier: budget (<$10), standard ($10-$18), premium (>$18)
    - calorie_category: light (<300), moderate (300-600), hearty (>600)
    - score_band: excellent (>=90), good (80-89), acceptable (70-79), failing (<70)
    - "healthy" = is_high_protein=true AND is_low_calorie=true
    - "needs_immediate_action" = severity='critical' OR deadline_days <= 7
    - For allergen filters use allergens.contains_* boolean columns (wheat, milk, egg, soy, peanut, tree_nut, fish, shellfish, sesame)
    - For top-N queries default to LIMIT 10"""

    MENU_TABLES = [
        f"{CATALOG}.menu_documents.silver_menu_items",
        f"{CATALOG}.menu_documents.silver_inspections",
        f"{CATALOG}.menu_documents.silver_violations",
        f"{CATALOG}.menu_documents.menu_items",
        f"{CATALOG}.menu_documents.nutritional_info",
        f"{CATALOG}.menu_documents.allergens",
        f"{CATALOG}.menu_documents.brand_nutrition_summary",
        f"{CATALOG}.menu_documents.inspection_details",
        f"{CATALOG}.menu_documents.violation_analysis",
        f"{CATALOG}.menu_documents.location_compliance_summary",
    ]

    MENU_QUESTIONS = [
        "Which menu items are gluten-free and under $15?",
        "What is the average calorie count per brand?",
        "Which locations have critical violations that need immediate action?",
        "Compare protein content across all burgers",
        "What is the latest inspection score for Chicago?",
        "Show me all premium-tier items that are also high protein",
        "Which brand has the most allergen-free options?",
        "What is the pass rate for each location?",
        "List all violations with urgency score above 5",
        "Which cuisine type has the lowest average calories?",
    ]

    MENU_EXAMPLE_SQLS = [
        {
            "question": "Gluten-free menu items under $15",
            "sql": f"""SELECT mi.brand_name, mi.item_name, mi.category, mi.price
    FROM {CATALOG}.menu_documents.menu_items mi
    JOIN {CATALOG}.menu_documents.allergens a
      ON mi.item_name = a.item_name AND mi.brand_name = a.brand_name
    WHERE a.contains_wheat = false AND mi.price < 15
    ORDER BY mi.price""",
        },
        {
            "question": "Average calories per brand",
            "sql": f"""SELECT brand_name,
           ROUND(AVG(calories), 0) AS avg_calories,
           MIN(calories) AS min_calories,
           MAX(calories) AS max_calories,
           COUNT(*) AS item_count
    FROM {CATALOG}.menu_documents.nutritional_info
    GROUP BY brand_name
    ORDER BY avg_calories DESC""",
        },
        {
            "question": "Pass rate by location",
            "sql": f"""SELECT location_name,
           total_inspections,
           passed_inspections,
           ROUND(pass_rate_pct, 1) AS pass_rate_pct,
           ROUND(avg_severity_index, 2) AS avg_severity
    FROM {CATALOG}.menu_documents.location_compliance_summary
    ORDER BY pass_rate_pct""",
        },
        {
            "question": "Critical violations needing immediate action, last 90 days",
            "sql": f"""SELECT location_name, inspection_date, code, severity, category, description, deadline_days, urgency_score
    FROM {CATALOG}.menu_documents.silver_violations
    WHERE needs_immediate_action = true
      AND inspection_date >= current_date() - 90
    ORDER BY urgency_score DESC, inspection_date DESC
    LIMIT 20""",
        },
        {
            "question": "Premium high-protein items",
            "sql": f"""SELECT n.brand_name, n.item_name, n.calories, n.protein_g, ROUND(n.protein_pct, 1) AS protein_pct, m.price
    FROM {CATALOG}.menu_documents.nutritional_info n
    JOIN {CATALOG}.menu_documents.menu_items m
      ON n.item_name = m.item_name AND n.brand_name = m.brand_name
    WHERE n.is_high_protein = true AND m.price_tier = 'premium'
    ORDER BY n.protein_g DESC""",
        },
        {
            "question": "Latest inspection score by location",
            "sql": f"""SELECT location_name, inspection_date, score, grade, score_band, violation_count, critical_count
    FROM {CATALOG}.menu_documents.silver_inspections
    QUALIFY ROW_NUMBER() OVER (PARTITION BY location_name ORDER BY inspection_date DESC) = 1
    ORDER BY score""",
        },
        {
            "question": "Cuisine types ranked by average calories",
            "sql": f"""SELECT mi.cuisine,
           ROUND(AVG(n.calories), 0) AS avg_calories,
           COUNT(*) AS items
    FROM {CATALOG}.menu_documents.menu_items mi
    JOIN {CATALOG}.menu_documents.nutritional_info n
      ON mi.item_name = n.item_name AND mi.brand_name = n.brand_name
    GROUP BY mi.cuisine
    ORDER BY avg_calories""",
        },
    ]

    MENU_JOIN_SPECS = [
        _build_join_spec(
            f"{CATALOG}.menu_documents.menu_items",
            f"{CATALOG}.menu_documents.nutritional_info",
            on=[("item_name", "item_name"), ("brand_name", "brand_name")],
            rt="FROM_RELATIONSHIP_TYPE_ONE_TO_ONE",
        ),
        _build_join_spec(
            f"{CATALOG}.menu_documents.menu_items",
            f"{CATALOG}.menu_documents.allergens",
            on=[("item_name", "item_name"), ("brand_name", "brand_name")],
            rt="FROM_RELATIONSHIP_TYPE_ONE_TO_ONE",
        ),
        _build_join_spec(
            f"{CATALOG}.menu_documents.nutritional_info",
            f"{CATALOG}.menu_documents.allergens",
            on=[("item_name", "item_name"), ("brand_name", "brand_name")],
            rt="FROM_RELATIONSHIP_TYPE_ONE_TO_ONE",
        ),
        _build_join_spec(
            f"{CATALOG}.menu_documents.silver_violations",
            f"{CATALOG}.menu_documents.silver_inspections",
            on=[("inspection_id", "inspection_id")],
        ),
        _build_join_spec(
            f"{CATALOG}.menu_documents.violation_analysis",
            f"{CATALOG}.menu_documents.inspection_details",
            on=[("inspection_id", "inspection_id")],
        ),
    ]

    # Benchmark questions — see REVENUE_BENCHMARKS for authoring notes.
    # Mix: 2 exact matches + 4 alternate phrasings.  Allergen and price-tier
    # benchmarks are the highest-value here because the General Instructions
    # document the `contains_*` boolean columns and `price_tier` derivations —
    # if Genie regresses on these, the instructions likely got truncated.
    MENU_BENCHMARKS = [
        {
            "question": "Which menu items are gluten-free and under $15?",
            "sql": f"""SELECT mi.brand_name, mi.item_name, mi.category, mi.price
    FROM {CATALOG}.menu_documents.menu_items mi
    JOIN {CATALOG}.menu_documents.allergens a
      ON mi.item_name = a.item_name AND mi.brand_name = a.brand_name
    WHERE a.contains_wheat = false AND mi.price < 15
    ORDER BY mi.price""",
        },
        {
            "question": "Which menu items don't contain wheat and cost less than $15?",
            "sql": f"""SELECT mi.brand_name, mi.item_name, mi.category, mi.price
    FROM {CATALOG}.menu_documents.menu_items mi
    JOIN {CATALOG}.menu_documents.allergens a
      ON mi.item_name = a.item_name AND mi.brand_name = a.brand_name
    WHERE a.contains_wheat = false AND mi.price < 15
    ORDER BY mi.price""",
        },
        {
            "question": "Average calorie count per brand",
            "sql": f"""SELECT brand_name,
           ROUND(AVG(calories), 0) AS avg_calories
    FROM {CATALOG}.menu_documents.nutritional_info
    GROUP BY brand_name
    ORDER BY avg_calories DESC""",
        },
        {
            "question": "What is the pass rate for each location?",
            "sql": f"""SELECT location_name, ROUND(pass_rate_pct, 1) AS pass_rate_pct
    FROM {CATALOG}.menu_documents.location_compliance_summary
    ORDER BY pass_rate_pct""",
        },
        {
            "question": "Critical violations needing immediate action in the last 90 days",
            "sql": f"""SELECT location_name, inspection_date, code, severity, category, description, deadline_days, urgency_score
    FROM {CATALOG}.menu_documents.silver_violations
    WHERE needs_immediate_action = true
      AND inspection_date >= current_date() - 90
    ORDER BY urgency_score DESC, inspection_date DESC
    LIMIT 20""",
        },
        {
            "question": "Which brand has the lowest average calories?",
            "sql": f"""SELECT brand_name,
           ROUND(AVG(calories), 0) AS avg_calories
    FROM {CATALOG}.menu_documents.nutritional_info
    GROUP BY brand_name
    ORDER BY avg_calories ASC
    LIMIT 1""",
        },
    ]

    menu_genie_id = create_or_get_genie(
        title=MENU_TITLE,
        description=MENU_DESCRIPTION,
        tables=MENU_TABLES,
        instructions_text=MENU_INSTRUCTIONS,
        example_sqls=MENU_EXAMPLE_SQLS,
        sample_questions=MENU_QUESTIONS,
        join_specs=MENU_JOIN_SPECS,
        benchmarks=MENU_BENCHMARKS,
    )
    print(f"Menu & Safety Genie ID: {menu_genie_id} ({len(MENU_BENCHMARKS)} benchmarks)")


In [ ]:
# Discover Domain tagging for created Genie spaces only.
import sys, os
sys.path.insert(0, os.path.abspath(".."))
from utils.domain_tags import ensure_domain_tag_policies, tag_workspace_entity

print("\n— Tagging Genie spaces for Discover Domains —")
ensure_domain_tag_policies(w, verbose=False)

_created_genies = [
    ("revenue", revenue_genie_id, "Revenue & Orders Intelligence", REVENUE_BENCHMARKS if "revenue" in GENIE_KEY_LIST else []),
    ("ops", ops_genie_id, "Operations Intelligence", OPS_BENCHMARKS if "ops" in GENIE_KEY_LIST else []),
    ("menu", menu_genie_id, "Menu & Safety Intelligence", MENU_BENCHMARKS if "menu" in GENIE_KEY_LIST else []),
]
for key, space_id, label, benchmarks in _created_genies:
    if not space_id:
        continue
    tag_workspace_entity(w, "geniespaces", space_id, GENIE_DOMAIN_TAGS[key], label=f"genie {label!r}")

print("\n\u2705 Genie Spaces stage complete")
_total_benchmarks = 0
for key, space_id, label, benchmarks in _created_genies:
    if space_id:
        print(f"   {label}: {space_id}  ({len(benchmarks)} benchmarks)")
        _total_benchmarks += len(benchmarks)
print(f"\n   Total benchmarks across {sum(1 for _, sid, _, _ in _created_genies if sid)} space(s): {_total_benchmarks}")
